In [0]:

from pyspark.sql.functions import col, explode_outer, input_file_name, current_timestamp, to_json, struct

In [0]:
df_raw = spark.read.option("multiLine", "true").json("/Volumes/workspace/bronze/fhir_raw/fhir")

In [0]:
df_resources = (
df_raw
.withColumn("entry_item", explode_outer(col("entry")))
.select(
    col("entry_item.resource").alias("resource"),
    col("_metadata.file_path").alias("source_file_name"),
    current_timestamp().alias("ingest_timestamp")
).withColumn("resource_type", col("resource.resourceType"))
)

In [0]:
df_resources.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.fhir_resources")